## 🎯 Learning Outcomes
### By the end of this module, you will be able to

- Understand the last two chapters of Python object-oriented programming:
  - **4 OOP Design Principles** — applying Encapsulation, Single Responsibility Principle (SRP), and Open/Closed Principle (OCP); understanding the benefits of composition over inheritance; and using `@dataclass` and `NamedTuple` for efficient data modeling, including `frozen`, `order`, `field`, and `__post_init__` features.
  - **5 Magic Methods (Dunder Methods)** — implementing various dunder methods to enable rich object behavior, such as:
    - Arithmetic operations (`__add__`, `__sub__`, `__mul__`, `__truediv__`, `__pow__`, `__matmul__`, `__neg__`)
    - Comparison operations (`__eq__`, `__lt__` with `@total_ordering`)
    - Container behaviors (`__len__`, `__getitem__`, `__setitem__`, `__contains__`, `__iter__`, `__next__`)
    - Context management (`__enter__`, `__exit__`)
    - Making objects callable (`__call__`)
    - Inspecting attributes with `__dict__` and `__class__`.

---

Attribute Introspection: `hasattr`, `getattr`, `setattr`

Python exposes three built-in functions that let you inspect and manipulate object attributes **dynamically** — without knowing their names at write-time.

| Function | Signature | What it does |
|---|---|---|
| `hasattr` | `hasattr(obj, name)` | Returns `True` if `obj` has attribute `name` |
| `getattr` | `getattr(obj, name[, default])` | Returns the value; raises `AttributeError` (or default) if missing |
| `setattr` | `setattr(obj, name, value)` | Sets (or creates) an attribute dynamically |

In [1]:
class Robot:
    def __init__(self, name: str, speed: int = 5):
        self.name = name
        self.speed = speed

    def greet(self):
        return f"Hi, I am {self.name}!"


r = Robot("Artoo", speed=8)

# ── hasattr ──────────────────────────────────────────────
print(hasattr(r, "name"))     # True  — attribute exists
print(hasattr(r, "colour"))   # False — attribute missing
print(hasattr(r, "greet"))    # True  — methods are attributes too!

True
False
True


In [2]:
# ── getattr ──────────────────────────────────────────────
print(getattr(r, "name"))                      # 'Artoo'
print(getattr(r, "colour", "silver"))          # 'silver'  — default used

# You can also call methods retrieved via getattr
method = getattr(r, "greet")
print(method())                                 # 'Hi, I am Artoo!'

Artoo
silver
Hi, I am Artoo!


In [3]:
# ── setattr ──────────────────────────────────────────────
setattr(r, "colour", "gold")   # creates a brand-new attribute
setattr(r, "speed", 10)        # overwrites an existing one

print(r.colour)   # 'gold'
print(r.speed)    # 10

gold
10


In [4]:
# ── Practical pattern: bulk-initialise from a dict ────────
config = {"name": "C3PO", "speed": 3, "language_count": 6_000_000}

bot = Robot.__new__(Robot)   # skip __init__
for key, value in config.items():
    setattr(bot, key, value)

print(vars(bot))   # shows all instance attributes as a dict

{'name': 'C3PO', 'speed': 3, 'language_count': 6000000}


In [5]:
# ── delattr ──────────────────────────────────────────────
# Bonus: delattr removes an attribute dynamically
delattr(bot, "language_count")
print(hasattr(bot, "language_count"))  # False

False


> **Tip:** `getattr` + `hasattr` are the backbone of frameworks (Django, Flask, pytest) that discover plugins, views, and fixtures by name at runtime.

---
## 4 · Encapsulation & Design Principles

### 4.1 Encapsulation — hiding implementation details

Encapsulation bundles data + behaviour together and **controls access** to internals.

| Convention | Syntax | Meaning |
|---|---|---|
| Public | `self.x` | Anyone may read/write |
| Protected (convention) | `self._x` | "Internal use" — don't touch from outside |
| Private (name-mangled) | `self.__x` | Becomes `_ClassName__x`; harder to access |

Python relies on **convention** + **properties**, not hard enforcement.

In [6]:
class BankAccount:
    def __init__(self, owner: str, balance: float = 0.0):
        self.owner = owner          # public
        self._balance = balance     # protected — internal use
        self.__pin = "1234"         # private — name-mangled

    # Property: controlled read access
    @property
    def balance(self) -> float:
        return self._balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Deposit must be positive.")
        self._balance += amount

    def withdraw(self, amount: float, pin: str) -> None:
        if pin != self.__pin:         # private checked internally
            raise PermissionError("Wrong PIN.")
        if amount > self._balance:
            raise ValueError("Insufficient funds.")
        self._balance -= amount


acct = BankAccount("Alice", 1000.0)
acct.deposit(500)
print(acct.balance)           # 1500.0  — via property

# acct._balance = -999        # works, but violates convention
# acct.__pin = "9999"         # creates a NEW attribute; doesn't change the real one!
print(acct._BankAccount__pin) # name-mangling revealed (avoid in real code)

1500.0
1234


### 4.2 SOLID Principles — overview

| Letter | Principle | One-liner |
|---|---|---|
| **S** | Single Responsibility | A class should have **one reason to change** |
| **O** | Open / Closed | Open for extension, **closed for modification** |
| **L** | Liskov Substitution | Subtypes must be substitutable for their base type |
| **I** | Interface Segregation | Many specific interfaces > one fat interface |
| **D** | Dependency Inversion | Depend on abstractions, not concretions |

In [7]:
# ── S: Single Responsibility ──────────────────────────────

# ❌ Violates SRP — class does TWO things
class OrderBad:
    def calculate_total(self, items): ...
    def send_email_confirmation(self, email): ...  # ← belongs elsewhere!

# ✅ Each class has one responsibility
class Order:
    def __init__(self, items: list):
        self.items = items

    def total(self) -> float:
        return sum(p for _, p in self.items)


class EmailService:
    def send_confirmation(self, email: str, order: Order) -> None:
        print(f"Sending confirmation to {email} for total ${order.total():.2f}")


order = Order([("Book", 15.00), ("Pen", 2.50)])
EmailService().send_confirmation("alice@example.com", order)

Sending confirmation to alice@example.com for total $17.50


In [8]:
# ── O: Open / Closed ──────────────────────────────────────
from abc import ABC, abstractmethod

class Discount(ABC):
    @abstractmethod
    def apply(self, price: float) -> float: ...

class NoDiscount(Discount):
    def apply(self, price: float) -> float:
        return price

class PercentDiscount(Discount):
    def __init__(self, pct: float): self.pct = pct
    def apply(self, price: float) -> float:
        return price * (1 - self.pct / 100)

# Add new discount types WITHOUT modifying existing code ✅
class BuyOneGetOne(Discount):
    def apply(self, price: float) -> float:
        return price / 2


for discount in [NoDiscount(), PercentDiscount(20), BuyOneGetOne()]:
    print(f"{type(discount).__name__:20s}: ${discount.apply(100):.2f}")

NoDiscount          : $100.00
PercentDiscount     : $80.00
BuyOneGetOne        : $50.00


### 4.3 Composition over Inheritance

> **"Favour object composition over class inheritance"** — GoF Design Patterns (1994)

Inheritance creates tight coupling. Composition **delegates** behaviour to injected objects — making code easier to test and extend.

In [9]:
# ── Inheritance approach (brittle) ────────────────────────
class Animal:
    def speak(self): return "..."

class Dog(Animal):
    def speak(self): return "Woof!"

class RobotDog(Dog):   # inherits 'speak' — but what if we want silent robot?
    pass


# ── Composition approach (flexible) ───────────────────────
class Voice:
    def speak(self) -> str: return "..."

class DogVoice(Voice):
    def speak(self) -> str: return "Woof!"

class SilentVoice(Voice):
    def speak(self) -> str: return "*silent*"


class Pet:
    def __init__(self, name: str, voice: Voice):
        self.name = name
        self._voice = voice              # ← composed, not inherited

    def speak(self) -> str:
        return f"{self.name} says: {self._voice.speak()}"


fido   = Pet("Fido",   DogVoice())
robodog = Pet("R2Dog", SilentVoice())

print(fido.speak())
print(robodog.speak())

Fido says: Woof!
R2Dog says: *silent*


### 4.4 Dataclasses — `@dataclass`

In [10]:
from dataclasses import dataclass, field

# Without dataclass — lots of boilerplate
class PointOld:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"
    def __eq__(self, other):
        return (self.x, self.y) == (other.x, other.y)


# With @dataclass — automatic __init__, __repr__, __eq__
@dataclass
class Point:
    x: float
    y: float

p1 = Point(1.0, 2.0)
p2 = Point(1.0, 2.0)
print(p1)           # Point(x=1.0, y=2.0)
print(p1 == p2)     # True — __eq__ generated automatically

Point(x=1.0, y=2.0)
True


In [11]:
# Advanced dataclass features
from dataclasses import dataclass, field

@dataclass(order=True, frozen=True)   # frozen = immutable (hashable)
class Vector:
    x: float = 0.0
    y: float = 0.0
    z: float = 0.0

    def magnitude(self) -> float:
        return (self.x**2 + self.y**2 + self.z**2) ** 0.5


@dataclass
class Inventory:
    items: list = field(default_factory=list)  # mutable default — safe!
    count: int = field(init=False, default=0)  # not in __init__

    def add(self, item: str):
        self.items.append(item)
        self.count += 1


v = Vector(3, 4, 0)
print(v, "| magnitude:", v.magnitude())

inv = Inventory()
inv.add("Sword")
inv.add("Shield")
print(inv)

Vector(x=3, y=4, z=0) | magnitude: 5.0
Inventory(items=['Sword', 'Shield'], count=2)


### 4.5 Named Tuples vs. Dataclasses

| Feature | `namedtuple` | `@dataclass` |
|---|---|---|
| Mutable | ❌ No | ✅ Yes (unless `frozen=True`) |
| Type hints | Optional | First-class |
| Methods | Limited | Full class methods |
| Memory | Slightly lighter | Slightly heavier |
| Hashable | ✅ Yes | Only if `frozen=True` |
| Iterable/unpackable | ✅ Yes | ❌ No |
| Best for | Lightweight read-only records | Richer data containers |

In [12]:
from collections import namedtuple
from typing import NamedTuple

# Classic namedtuple
Color = namedtuple("Color", ["r", "g", "b"])

# Typed NamedTuple (modern, preferred)
class RGBColor(NamedTuple):
    r: int
    g: int
    b: int
    alpha: float = 1.0   # default value

    def to_hex(self) -> str:
        return f"#{self.r:02x}{self.g:02x}{self.b:02x}"


red = RGBColor(255, 0, 0)
print(red)
print(red.to_hex())
print(f"Unpackable: {(*red,)}")     # ← namedtuple advantage
r, g, b, a = red                    # tuple unpacking works!
print(r, g, b, a)

RGBColor(r=255, g=0, b=0, alpha=1.0)
#ff0000
Unpackable: (255, 0, 0, 1.0)
255 0 0 1.0


In [13]:
# When to choose which?
from dataclasses import dataclass

# Use NamedTuple when: small, immutable, unpack-able, used as dict keys
class Coordinate(NamedTuple):
    lat: float
    lon: float

# Use dataclass when: you need mutability, validation, rich methods, or post-init logic
@dataclass
class UserProfile:
    username: str
    email: str
    score: int = 0

    def __post_init__(self):  # runs after __init__
        self.email = self.email.lower()  # normalise email
        if not self.username:
            raise ValueError("Username cannot be empty.")


u = UserProfile("Alice", "Alice@EXAMPLE.COM")
print(u)   # email is normalised

UserProfile(username='Alice', email='alice@example.com', score=0)


---
## 5 · Magic Methods (Dunder Methods)

> **Dunder** = **d**ouble **under**score. These `__special__` methods let your objects integrate seamlessly with Python's syntax and built-in functions.

### 5.1 Arithmetic Dunders

In [14]:
class Fraction:
    """Simplified fraction supporting arithmetic dunders."""

    def __init__(self, numerator: int, denominator: int):
        if denominator == 0:
            raise ZeroDivisionError("Denominator cannot be zero.")
        from math import gcd
        g = gcd(abs(numerator), abs(denominator))
        self.num = numerator   // g
        self.den = denominator // g

    def __repr__(self) -> str:
        return f"{self.num}/{self.den}"

    # Arithmetic
    def __add__(self, other: "Fraction") -> "Fraction":
        return Fraction(self.num * other.den + other.num * self.den,
                        self.den * other.den)

    def __sub__(self, other: "Fraction") -> "Fraction":
        return Fraction(self.num * other.den - other.num * self.den,
                        self.den * other.den)

    def __mul__(self, other: "Fraction") -> "Fraction":
        return Fraction(self.num * other.num, self.den * other.den)

    def __truediv__(self, other: "Fraction") -> "Fraction":
        return Fraction(self.num * other.den, self.den * other.num)

    def __pow__(self, exp: int) -> "Fraction":
        return Fraction(self.num ** exp, self.den ** exp)


a = Fraction(1, 2)
b = Fraction(1, 3)

print(f"{a} + {b} = {a + b}")
print(f"{a} - {b} = {a - b}")
print(f"{a} * {b} = {a * b}")
print(f"{a} / {b} = {a / b}")
print(f"{a} ** 3 = {a ** 3}")

1/2 + 1/3 = 5/6
1/2 - 1/3 = 1/6
1/2 * 1/3 = 1/6
1/2 / 1/3 = 3/2
1/2 ** 3 = 1/8


### 5.2 Comparison Dunders

In [15]:
from functools import total_ordering

@total_ordering   # generates the rest from __eq__ + __lt__
class Temperature:
    def __init__(self, celsius: float):
        self.celsius = celsius

    def __repr__(self): return f"{self.celsius}°C"

    # Required by @total_ordering
    def __eq__(self, other) -> bool:
        return self.celsius == other.celsius

    def __lt__(self, other) -> bool:
        return self.celsius < other.celsius

    # @ total_ordering auto-generates: __le__, __gt__, __ge__


temps = [Temperature(100), Temperature(-10), Temperature(37), Temperature(0)]
print("Sorted:", sorted(temps))
print("Min:", min(temps))
print("Max:", max(temps))

t1, t2 = Temperature(20), Temperature(30)
print(f"{t1} == {t2}: {t1 == t2}")
print(f"{t1} <  {t2}: {t1 < t2}")
print(f"{t1} >= {t2}: {t1 >= t2}")

Sorted: [-10°C, 0°C, 37°C, 100°C]
Min: -10°C
Max: 100°C
20°C == 30°C: False
20°C <  30°C: True
20°C >= 30°C: False


### 5.3 Container Dunders

These make your objects behave like sequences or mappings.

In [16]:
class WordBag:
    """A case-insensitive bag (multiset) of words."""

    def __init__(self, *words):
        self._data: dict[str, int] = {}
        for w in words:
            self._add(w)

    def _add(self, word: str):
        key = word.lower()
        self._data[key] = self._data.get(key, 0) + 1

    # ── Container protocol ───────────────────────────────
    def __len__(self) -> int:
        return sum(self._data.values())

    def __contains__(self, word: str) -> bool:   # enables `in` operator
        return word.lower() in self._data

    def __getitem__(self, word: str) -> int:      # enables bag["hello"]
        return self._data.get(word.lower(), 0)

    def __setitem__(self, word: str, count: int): # bag["hello"] = 5
        self._data[word.lower()] = count

    def __delitem__(self, word: str):             # del bag["hello"]
        del self._data[word.lower()]

    def __iter__(self):                           # enables for-loop
        return iter(self._data)

    def __repr__(self):
        return f"WordBag({self._data})"


bag = WordBag("apple", "Banana", "apple", "cherry", "Apple")

print("Length:",      len(bag))
print("Count apple:", bag["apple"])
print("Has banana:",  "banana" in bag)
print("Has grape:",   "grape"  in bag)

bag["grape"] = 3
del bag["cherry"]

for word in bag:
    print(f"  {word}: {bag[word]}")

Length: 5
Count apple: 3
Has banana: True
Has grape: False
  apple: 3
  banana: 1
  grape: 3


In [17]:
# ── __iter__ and __next__: building a custom iterator ────
class Countdown:
    """Counts down from n to 0."""

    def __init__(self, n: int):
        self.n = n
        self._current = n

    def __iter__(self):
        self._current = self.n   # reset on each iteration
        return self

    def __next__(self) -> int:
        if self._current < 0:
            raise StopIteration
        val = self._current
        self._current -= 1
        return val


cd = Countdown(5)
print(list(cd))          # [5, 4, 3, 2, 1, 0]
print(list(cd))          # resets: [5, 4, 3, 2, 1, 0]

for n in Countdown(3):
    print(n, end=" ")    # 3 2 1 0
print()

[5, 4, 3, 2, 1, 0]
[5, 4, 3, 2, 1, 0]
3 2 1 0 


### 5.4 Context Manager Dunders: `__enter__` & `__exit__`

In [18]:
import time

class Timer:
    """Context manager that measures elapsed time."""

    def __enter__(self):
        self._start = time.perf_counter()
        return self                        # bound to `as` variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self._start
        print(f"Elapsed: {self.elapsed:.4f}s")
        return False   # False = don't suppress exceptions


with Timer() as t:
    total = sum(range(1_000_000))

print(f"Sum = {total}, time = {t.elapsed:.4f}s")

Elapsed: 0.0519s
Sum = 499999500000, time = 0.0519s


In [19]:
# Exception handling in __exit__
class Suppressor:
    """Suppresses a specific exception type."""

    def __init__(self, *exception_types):
        self.exception_types = exception_types

    def __enter__(self): return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type and issubclass(exc_type, self.exception_types):
            print(f"Suppressed: {exc_type.__name__}: {exc_val}")
            return True   # suppress the exception
        return False


with Suppressor(ZeroDivisionError):
    x = 1 / 0   # would normally crash

print("Code continues after suppressed exception!")

Suppressed: ZeroDivisionError: division by zero
Code continues after suppressed exception!


### 5.5 Callable Objects: `__call__`

In [20]:
class Multiplier:
    """A callable object — instances work like functions."""

    def __init__(self, factor: float):
        self.factor = factor

    def __call__(self, value: float) -> float:
        return value * self.factor

    def __repr__(self):
        return f"Multiplier(×{self.factor})"


double = Multiplier(2)
triple = Multiplier(3)

print(double(10))     # 20.0
print(triple(10))     # 30.0
print(callable(double))  # True

# Useful for: stateful callbacks, decorators-as-classes, memoisation
numbers = [1, 2, 3, 4, 5]
print(list(map(double, numbers)))   # [2, 4, 6, 8, 10]

20
30
True
[2, 4, 6, 8, 10]


In [21]:
# Practical: a class-based memoisation decorator
class Memoize:
    def __init__(self, func):
        self.func  = func
        self.cache: dict = {}

    def __call__(self, *args):
        if args not in self.cache:
            self.cache[args] = self.func(*args)
        return self.cache[args]

    def __repr__(self):
        return f"Memoize({self.func.__name__}, cached={list(self.cache.keys())})"


@Memoize
def fibonacci(n: int) -> int:
    if n < 2: return n
    return fibonacci(n - 1) + fibonacci(n - 2)


print([fibonacci(i) for i in range(10)])
print(fibonacci)

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
Memoize(fibonacci, cached=[(0,), (1,), (2,), (3,), (4,), (5,), (6,), (7,), (8,), (9,)])


### 5.6 `__slots__`, `__dict__`, `__class__`

In [22]:
# ── __dict__ ─────────────────────────────────────────────
class Dog:
    def __init__(self, name, breed):
        self.name  = name
        self.breed = breed

d = Dog("Rex", "Labrador")
print(d.__dict__)        # {'name': 'Rex', 'breed': 'Labrador'}
print(Dog.__dict__.keys())  # class-level attributes & methods

{'name': 'Rex', 'breed': 'Labrador'}
dict_keys(['__module__', '__init__', '__dict__', '__weakref__', '__doc__'])


In [23]:
# ── __class__ ────────────────────────────────────────────
print(d.__class__)            # <class '__main__.Dog'>
print(d.__class__.__name__)   # 'Dog'
print(isinstance(d, d.__class__))  # True

<class '__main__.Dog'>
Dog
True


In [24]:
import sys

# ── __slots__ — memory optimisation ──────────────────────
class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedPoint:
    __slots__ = ("x", "y")   # no __dict__; only these attributes allowed

    def __init__(self, x, y):
        self.x = x
        self.y = y


rp = RegularPoint(1, 2)
sp = SlottedPoint(1, 2)

print("Regular size:", sys.getsizeof(rp) + sys.getsizeof(rp.__dict__), "bytes")
print("Slotted size:", sys.getsizeof(sp), "bytes")

# Slotted objects are faster to create and use less memory
# Trade-off: can't add arbitrary new attributes
try:
    sp.z = 3
except AttributeError as e:
    print(f"Error: {e}")

Regular size: 344 bytes
Slotted size: 48 bytes
Error: 'SlottedPoint' object has no attribute 'z'


In [25]:
# Benchmark: slots vs regular (creating 1 million objects)
import timeit

regular = timeit.timeit(lambda: RegularPoint(1, 2), number=1_000_000)
slotted = timeit.timeit(lambda: SlottedPoint(1, 2), number=1_000_000)

print(f"Regular: {regular:.3f}s")
print(f"Slotted: {slotted:.3f}s")
print(f"Speed-up: {regular/slotted:.2f}×")

Regular: 0.486s
Slotted: 0.478s
Speed-up: 1.02×


---
## 5.7 · Practical Example: Custom `Matrix` Class with Full Operator Overloading

We now build a **production-quality** `Matrix` class that implements:
- `__repr__` / `__str__`
- Arithmetic: `+`, `-`, `*` (scalar & matrix), `/` (scalar)
- Comparison: `==`, `!=`
- Container: `len`, `[]`, `in`, iteration
- Context manager for timing
- `__call__` for applying a function element-wise

In [26]:
from __future__ import annotations
from typing import Union, Callable, Iterator
import copy


class Matrix:
    """
    A 2-D numerical matrix with full operator overloading.

    Supports:
      - Arithmetic: +, -, *, /, ** (scalar or element-wise)
      - Comparison: ==, !=
      - Indexing:   mat[row, col], mat[row]
      - Iteration:  iterates over rows
      - Callable:   mat(fn) applies fn element-wise
      - Context manager for timing a block
    """

    __slots__ = ("_data", "rows", "cols", "_start", "elapsed")

    # ──────────────────────────────────────────────────────
    # Construction helpers
    # ──────────────────────────────────────────────────────
    def __init__(self, data: list[list[float]]):
        if not data or not data[0]:
            raise ValueError("Matrix must be non-empty.")
        n_cols = len(data[0])
        if any(len(row) != n_cols for row in data):
            raise ValueError("All rows must have the same length.")
        self._data = [list(row) for row in data]   # deep copy
        self.rows  = len(data)
        self.cols  = n_cols
        # Initialize _start and elapsed for __slots__ consistency
        self._start = 0.0
        self.elapsed = 0.0

    @classmethod
    def zeros(cls, rows: int, cols: int) -> Matrix:
        return cls([[0.0] * cols for _ in range(rows)])

    @classmethod
    def identity(cls, n: int) -> Matrix:
        m = cls.zeros(n, n)
        for i in range(n):
            m._data[i][i] = 1.0
        return m

    # ──────────────────────────────────────────────────────
    # Representation
    # ──────────────────────────────────────────────────────
    def __repr__(self) -> str:
        rows_str = ", ".join(str(row) for row in self._data)
        return f"Matrix([{rows_str}])"

    def __str__(self) -> str:
        width = max(len(str(v)) for row in self._data for v in row)
        lines = ["  [" + "  ".join(f"{v:{width}}" for v in row) + "]" for row in self._data]
        return "[\n" + "\n".join(lines) + "\n]"

    # ──────────────────────────────────────────────────────
    # Container dunders
    # ──────────────────────────────────────────────────────
    def __len__(self) -> int:
        """Number of rows."""
        return self.rows

    def __getitem__(self, idx):
        """mat[i] → row i; mat[i, j] → element (i, j)."""
        if isinstance(idx, tuple):
            r, c = idx
            return self._data[r][c]
        return self._data[idx]

    def __setitem__(self, idx, value):
        """mat[i, j] = v"""
        if isinstance(idx, tuple):
            r, c = idx
            self._data[r][c] = value
        else:
            self._data[idx] = list(value)

    def __contains__(self, value) -> bool:
        """value in mat → True if value appears anywhere."""
        return any(value in row for row in self._data)

    def __iter__(self) -> Iterator[list]:
        """Iterates over rows."""
        return iter(self._data)

    # ──────────────────────────────────────────────────────
    # Arithmetic dunders
    # ──────────────────────────────────────────────────────
    def _check_shape(self, other: Matrix, op: str) -> None:
        if self.rows != other.rows or self.cols != other.cols:
            raise ValueError(f"{op} requires matching shapes: "
                             f"{self.rows}×{self.cols} vs {other.rows}×{other.cols}")

    def __add__(self, other: Union[Matrix, float]) -> Matrix:
        if isinstance(other, Matrix):
            self._check_shape(other, "Addition")
            return Matrix([[self._data[i][j] + other._data[i][j]
                            for j in range(self.cols)] for i in range(self.rows)])
        return Matrix([[v + other for v in row] for row in self._data])

    def __radd__(self, other: float) -> Matrix:
        return self.__add__(other)

    def __sub__(self, other: Union[Matrix, float]) -> Matrix:
        if isinstance(other, Matrix):
            self._check_shape(other, "Subtraction")
            return Matrix([[self._data[i][j] - other._data[i][j]
                            for j in range(self.cols)] for i in range(self.rows)])
        return Matrix([[v - other for v in row] for row in self._data])

    def __mul__(self, other: Union[Matrix, float]) -> Matrix:
        """Element-wise multiply (Hadamard) or scalar multiply."""
        if isinstance(other, Matrix):
            self._check_shape(other, "Element-wise multiplication")
            return Matrix([[self._data[i][j] * other._data[i][j]
                            for j in range(self.cols)] for i in range(self.rows)])
        return Matrix([[v * other for v in row] for row in self._data])

    def __rmul__(self, other: float) -> Matrix:
        return self.__mul__(other)

    def __truediv__(self, other: float) -> Matrix:
        return Matrix([[v / other for v in row] for row in self._data])

    def __pow__(self, exp: int) -> Matrix:
        """Matrix exponentiation (repeated @ for integer powers)."""
        if self.rows != self.cols:
            raise ValueError("Matrix must be square for exponentiation.")
        result = Matrix.identity(self.rows)
        base   = copy.deepcopy(self)
        for _ in range(exp):
            result = result @ base
        return result

    def __matmul__(self, other: Matrix) -> Matrix:
        """True matrix multiplication via @ operator."""
        if self.cols != other.rows:
            raise ValueError(f"Cannot multiply {self.rows}×{self.cols} @ {other.rows}×{other.cols}")
        result = Matrix.zeros(self.rows, other.cols)
        for i in range(self.rows):
            for j in range(other.cols):
                result[i, j] = sum(self._data[i][k] * other._data[k][j]
                                   for k in range(self.cols))
        return result

    def __neg__(self) -> Matrix:
        return self * -1

    # ──────────────────────────────────────────────────────
    # Comparison dunders
    # ──────────────────────────────────────────────────────
    def __eq__(self, other) -> bool:
        if not isinstance(other, Matrix): return NotImplemented
        return self._data == other._data

    def __ne__(self, other) -> bool:
        return not self.__eq__(other)

    # ──────────────────────────────────────────────────────
    # Callable — apply a function element-wise
    # ──────────────────────────────────────────────────────
    def __call__(self, fn: Callable[[float], float]) -> Matrix:
        """mat(fn) → new matrix with fn applied to every element."""
        return Matrix([[fn(v) for v in row] for row in self._data])

    # ──────────────────────────────────────────────────────
    # Context manager — measure operation time
    # ──────────────────────────────────────────────────────
    def __enter__(self):
        import time
        self._start = time.perf_counter()
        return self

    def __exit__(self, *args):
        import time
        self.elapsed = time.perf_counter() - self._start
        print(f"Matrix operation took {self.elapsed*1000:.3f} ms")
        return False

    # ──────────────────────────────────────────────────────
    # Utility methods
    # ──────────────────────────────────────────────────────
    def transpose(self) -> Matrix:
        return Matrix([[self._data[i][j] for i in range(self.rows)]
                       for j in range(self.cols)])

    def trace(self) -> float:
        if self.rows != self.cols:
            raise ValueError("Trace is defined for square matrices only.")
        return sum(self._data[i][i] for i in range(self.rows))


print("Matrix class defined ✓")

Matrix class defined ✓


In [27]:
# ── Demo: basic creation and representation ───────────────
A = Matrix([[1, 2, 3],
            [4, 5, 6],
            [7, 8, 9]])

I = Matrix.identity(3)
Z = Matrix.zeros(2, 3)

print("Matrix A:")
print(A)
print("\nIdentity 3×3:")
print(I)

Matrix A:
[
  [1  2  3]
  [4  5  6]
  [7  8  9]
]

Identity 3×3:
[
  [1.0  0.0  0.0]
  [0.0  1.0  0.0]
  [0.0  0.0  1.0]
]


In [28]:
# ── Demo: arithmetic ─────────────────────────────────────
B = Matrix([[9, 8, 7],
            [6, 5, 4],
            [3, 2, 1]])

print("A + B:");    print(A + B)
print("A - B:");    print(A - B)
print("A * 2:");    print(A * 2)    # scalar
print("3 * A:");    print(3 * A)    # __rmul__
print("A * B (element-wise):"); print(A * B)

A + B:
[
  [10  10  10]
  [10  10  10]
  [10  10  10]
]
A - B:
[
  [-8  -6  -4]
  [-2   0   2]
  [ 4   6   8]
]
A * 2:
[
  [ 2   4   6]
  [ 8  10  12]
  [14  16  18]
]
3 * A:
[
  [ 3   6   9]
  [12  15  18]
  [21  24  27]
]
A * B (element-wise):
[
  [ 9  16  21]
  [24  25  24]
  [21  16   9]
]


In [29]:
# ── Demo: matrix multiplication and power ────────────────
M = Matrix([[1, 2],
            [3, 4]])

N = Matrix([[5, 6],
            [7, 8]])

print("M @ N (matrix mul):"); print(M @ N)
print("M ** 3:"); print(M ** 3)

M @ N (matrix mul):
[
  [19  22]
  [43  50]
]
M ** 3:
[
  [ 37.0   54.0]
  [ 81.0  118.0]
]


In [30]:
# ── Demo: container dunders ──────────────────────────────
print("len(A):", len(A))           # number of rows
print("A[1]:",   A[1])             # row 1
print("A[0,2]:", A[0, 2])         # element at (0, 2)
print("5 in A:", 5 in A)          # __contains__
print("99 in A:", 99 in A)

print("\nIterating over rows:")
for row in A:
    print(" ", row)

len(A): 3
A[1]: [4, 5, 6]
A[0,2]: 3
5 in A: True
99 in A: False

Iterating over rows:
  [1, 2, 3]
  [4, 5, 6]
  [7, 8, 9]


In [31]:
# ── Demo: __call__ — apply ReLU activation element-wise ──
import math

raw = Matrix([[-3, -1,  0],
              [ 2,  4, -2]])

relu    = raw(lambda x: max(0, x))
sigmoid = raw(lambda x: 1 / (1 + math.exp(-x)))

print("ReLU applied:");    print(relu)
print("Sigmoid applied:"); print(sigmoid)

ReLU applied:
[
  [0  0  0]
  [2  4  0]
]
Sigmoid applied:
[
  [0.04742587317756678   0.2689414213699951                  0.5]
  [ 0.8807970779778823   0.9820137900379085  0.11920292202211755]
]


In [32]:
# ── Demo: context manager ────────────────────────────────
big_A = Matrix([[i * j for j in range(1, 51)] for i in range(1, 51)])
big_B = Matrix([[i + j for j in range(1, 51)] for i in range(1, 51)])

with big_A:   # times the block
    result = big_A @ big_B

print(f"Result shape: {result.rows}×{result.cols}")

Matrix operation took 45.788 ms
Result shape: 50×50


In [33]:
# ── Demo: comparison ─────────────────────────────────────
X = Matrix([[1, 2], [3, 4]])
Y = Matrix([[1, 2], [3, 4]])
Z2 = Matrix([[1, 2], [3, 5]])

print("X == Y:", X == Y)    # True
print("X == Z:", X == Z2)   # False
print("X != Z:", X != Z2)   # True

X == Y: True
X == Z: False
X != Z: True


In [34]:
# ── Demo: transpose and trace ────────────────────────────
P = Matrix([[1, 2, 3],
            [4, 5, 6]])

print("P:");           print(P)
print("P.transpose():"); print(P.transpose())

Q = Matrix([[1, 0, 0],
            [0, 2, 0],
            [0, 0, 3]])
print("Trace of Q:", Q.trace())   # 6

P:
[
  [1  2  3]
  [4  5  6]
]
P.transpose():
[
  [1  4]
  [2  5]
  [3  6]
]
Trace of Q: 6


---
## Summary

| Topic | Key takeaway |
|---|---|
| `hasattr` / `getattr` / `setattr` | Dynamic attribute access — backbone of frameworks |
| Encapsulation | `_` = convention; `__` = name-mangled; `@property` for controlled access |
| SOLID (S & O) | One responsibility per class; extend without modifying |
| Composition | Inject collaborators instead of inheriting from them |
| `@dataclass` | Auto-generates `__init__`, `__repr__`, `__eq__`; supports `frozen`, `order`, `post_init` |
| `NamedTuple` | Immutable, unpackable, hashable — great for lightweight records |
| Arithmetic dunders | `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__pow__`, `__matmul__` |
| Comparison dunders | `__eq__`, `__lt__` + `@total_ordering` generates the rest |
| Container dunders | `__len__`, `__getitem__`, `__setitem__`, `__contains__`, `__iter__`, `__next__` |
| Context manager | `__enter__` / `__exit__` for resource management and timing |
| `__call__` | Makes instances callable — powerful for stateful callbacks |
| `__slots__` | Saves memory, speeds up attribute access, prevents arbitrary attributes |
| `Matrix` class | Shows all of the above working together in a real-world example |

### Contributed by: Abdulhadi Zubailah